In [ ]:
# Install required dependencies
!pip install -q kaggle-benchmarks numpy

# Near vs. Far Transfer BenchmarkTests generalization across similarity distances.**Cognitive Science**: Thorndike & Woodworth (1901), Barnett & Ceci (2002)

In [ ]:
"""Novel Rule System Generator for Learning Benchmarks.Generates procedural rule systems that cannot be in training data.Each system defines a mapping from inputs to outputs via a chainof deterministic rules. Difficulty is controlled by:- Number of rules- Number of input features- Rule interaction complexity (independent vs. chained)Systems are seeded for reproducibility across runs."""import randomimport hashlibfrom dataclasses import dataclass, field@dataclassclass RuleSystem:    """A generated rule system with examples."""    name: str    description: str    rules: list[str]    examples: list[dict]  # {"input": str, "output": str}    test_items: list[dict]  # {"input": str, "output": str}    difficulty: int  # 1-3    n_rules: int    domain: str  # "symbol", "language", "number"def _make_rng(seed: str) -> random.Random:    h = int(hashlib.sha256(seed.encode()).hexdigest(), 16)    return random.Random(h)def generate_symbol_system(seed: str = "sym_default", difficulty: int = 1) -> RuleSystem:    """    Generate a symbol transformation rule system.    Input: sequence of symbols (e.g., "△ ○ □")    Rules: transformations (e.g., "△ followed by ○ becomes ★")    Output: transformed sequence    """    rng = _make_rng(seed)    shapes = ["△", "○", "□", "◇", "★", "⬡", "⬟", "▽"]    colors = ["red", "blue", "green", "yellow"]    if difficulty == 1:        # Simple 1-to-1 substitution        src = rng.sample(shapes[:4], 3)        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes[4:])]        mapping = dict(zip(src, dst[:3]))        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]        rules.append("All other symbols stay the same")        def apply_rules(seq):            return [mapping.get(s, s) for s in seq]    elif difficulty == 2:        # Context-dependent: pairs matter        src = rng.sample(shapes[:5], 4)        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes)]        mapping = dict(zip(src[:3], dst[:3]))        pair_rule = (src[0], src[1], dst[3])  # "X followed by Y becomes Z"        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]        rules.append(f"EXCEPTION: {pair_rule[0]} followed by {pair_rule[1]} → both become {pair_rule[2]}")        rules.append("All other symbols stay the same")        def apply_rules(seq):            result = []            i = 0            while i < len(seq):                if i + 1 < len(seq) and seq[i] == pair_rule[0] and seq[i + 1] == pair_rule[1]:                    result.extend([pair_rule[2], pair_rule[2]])                    i += 2                else:                    result.append(mapping.get(seq[i], seq[i]))                    i += 1            return result    else:  # difficulty == 3        # Multi-pass with conditional rules        src = rng.sample(shapes[:6], 5)        dst = rng.sample(shapes, 5)        mapping1 = {src[0]: dst[0], src[1]: dst[1]}        mapping2 = {dst[0]: dst[2]}  # Chain: src[0] → dst[0] → dst[2]        cond = src[2]  # If this symbol is present, apply extra rule        extra_map = {src[3]: dst[3]}        rules = [            f"Pass 1: Replace {s} with {d}" for s, d in mapping1.items()        ]        rules.append(f"Pass 2: Replace {list(mapping2.keys())[0]} with {list(mapping2.values())[0]}")        rules.append(f"IF the sequence contains {cond}: also replace {src[3]} with {dst[3]}")        rules.append("All other symbols stay the same throughout")        def apply_rules(seq):            # Pass 1            result = [mapping1.get(s, s) for s in seq]            # Pass 2            result = [mapping2.get(s, s) for s in result]            # Conditional            if cond in seq:  # Check original sequence                result = [extra_map.get(s, s) for s in result]            return result    # Generate examples    all_items = []    for _ in range(25):        length = rng.randint(3, 6)        seq = [rng.choice(shapes[:5]) for _ in range(length)]        output = apply_rules(seq)        all_items.append({"input": " ".join(seq), "output": " ".join(output)})    # Deduplicate by input    seen = set()    unique_items = []    for item in all_items:        if item["input"] not in seen:            seen.add(item["input"])            unique_items.append(item)    rng.shuffle(unique_items)    n_examples = min(15, len(unique_items) - 5)    examples = unique_items[:n_examples]    test_items = unique_items[n_examples:n_examples + 5]    return RuleSystem(        name=f"SymbolTransform-{seed}",        description="Apply symbol transformation rules to input sequences",        rules=rules,        examples=examples,        test_items=test_items,        difficulty=difficulty,        n_rules=len(rules),        domain="symbol",    )def generate_number_system(seed: str = "num_default", difficulty: int = 1) -> RuleSystem:    """    Generate a novel number system / arithmetic.    Input: expression in the invented system    Rules: how operators work    Output: numeric result    """    rng = _make_rng(seed)    op_names = ["grok", "flim", "zorp", "quex", "blix"]    ops = rng.sample(op_names, 3)    if difficulty == 1:        # Two operators: basic arithmetic with twist        a_op, b_op = ops[0], ops[1]        a_fn = lambda x, y: x + y + 1  # "grok" = add and increment        b_fn = lambda x, y: abs(x - y)  # "flim" = absolute difference        rules = [            f"'{a_op}(x, y)' means: add x and y, then add 1",            f"'{b_op}(x, y)' means: absolute difference of x and y",        ]        op_map = {a_op: a_fn, b_op: b_fn}    elif difficulty == 2:        a_op, b_op, c_op = ops[0], ops[1], ops[2]        a_fn = lambda x, y: x * 2 + y        b_fn = lambda x, y: (x + y) % 10        c_fn = lambda x, y: max(x, y) - min(x, y) + 1        rules = [            f"'{a_op}(x, y)' means: double x, then add y",            f"'{b_op}(x, y)' means: add x and y, take the last digit (mod 10)",            f"'{c_op}(x, y)' means: difference of larger and smaller, plus 1",        ]        op_map = {a_op: a_fn, b_op: b_fn, c_op: c_fn}    else:  # difficulty == 3        a_op, b_op, c_op = ops[0], ops[1], ops[2]        # Nested operations        a_fn = lambda x, y: x + y + 1        b_fn = lambda x, y: x * y        rules = [            f"'{a_op}(x, y)' means: add x and y, then add 1",            f"'{b_op}(x, y)' means: multiply x and y",            f"Operations can be nested: '{a_op}({b_op}(x, y), z)' means: first compute {b_op}(x, y), then use the result as the first argument to {a_op}",        ]        op_map = {a_op: a_fn, b_op: b_fn}    # Generate examples    all_items = []    for _ in range(20):        if difficulty <= 2:            op_name = rng.choice(list(op_map.keys()))            x = rng.randint(1, 9)            y = rng.randint(1, 9)            result = op_map[op_name](x, y)            expr = f"{op_name}({x}, {y})"        else:            # Allow nesting            if rng.random() < 0.5:                op_name = rng.choice(list(op_map.keys()))                x = rng.randint(1, 9)                y = rng.randint(1, 9)                result = op_map[op_name](x, y)                expr = f"{op_name}({x}, {y})"            else:                inner_op = rng.choice(list(op_map.keys()))                outer_op = rng.choice(list(op_map.keys()))                x, y, z = rng.randint(1, 5), rng.randint(1, 5), rng.randint(1, 5)                inner_result = op_map[inner_op](x, y)                result = op_map[outer_op](inner_result, z)                expr = f"{outer_op}({inner_op}({x}, {y}), {z})"        all_items.append({"input": expr, "output": str(result)})    # Deduplicate    seen = set()    unique_items = []    for item in all_items:        if item["input"] not in seen:            seen.add(item["input"])            unique_items.append(item)    rng.shuffle(unique_items)    n_ex = min(12, len(unique_items) - 5)    examples = unique_items[:n_ex]    test_items = unique_items[n_ex:n_ex + 5]    return RuleSystem(        name=f"NumberSystem-{seed}",        description="Evaluate expressions using novel arithmetic operators",        rules=rules,        examples=examples,        test_items=test_items,        difficulty=difficulty,        n_rules=len(rules),        domain="number",    )# Pre-generated systems for the benchmarkLEARNING_CURVE_SYSTEMS = [    generate_symbol_system("lc_sym_easy", difficulty=1),    generate_symbol_system("lc_sym_med", difficulty=2),    generate_symbol_system("lc_sym_hard", difficulty=3),    generate_number_system("lc_num_easy", difficulty=1),    generate_number_system("lc_num_med", difficulty=2),    generate_number_system("lc_num_hard", difficulty=3),]# Systems for transfer testingTRANSFER_BASE_SYSTEM = generate_symbol_system("transfer_base", difficulty=2)TRANSFER_NEAR_SYSTEM = generate_symbol_system("transfer_near", difficulty=2)TRANSFER_FAR_SYSTEM = generate_number_system("transfer_far", difficulty=2)# Systems for interference testingINTERFERENCE_A = generate_symbol_system("interf_a", difficulty=2)INTERFERENCE_B = generate_symbol_system("interf_b_similar", difficulty=2)

In [ ]:
"""Learning Benchmark 2: Near vs. Far TransferTests whether models can generalize learned rules to novel contexts.Near transfer: same structure, different surface features.Far transfer: same principle, completely different domain.Cognitive Science Basis:- Thorndike & Woodworth (1901): Transfer of practice- Barnett & Ceci (2002): Taxonomy of far transfer- Genuine learning should show some transfer; pure memorization shows none.Score: Weighted transfer performance across distances."""import kaggle_benchmarks as kbenchfrom dataclasses import dataclassimport numpy as npimport reimport json# Rule system generators defined above@dataclassclass TransferAnswer:    answer: str    reasoning: strdef normalize_output(text: str) -> str:    text = text.strip().lower()    text = re.sub(r'\s+', ' ', text)    return textdef check_output(model_output: str, expected: str) -> bool:    m = normalize_output(model_output)    e = normalize_output(expected)    return e in m or m in e# ── Transfer Test Sets ──# Training system: symbol transform (difficulty 2)# Near transfer: same type (symbol) with different symbols# Far transfer: number system with structurally analogous rulesTRAIN_SYSTEM = generate_symbol_system("transfer_train_v2", difficulty=2)NEAR_SYSTEM = generate_symbol_system("transfer_near_v2", difficulty=2)# For far transfer: we teach the symbol system, then test on number# system that has analogous structure but different domainFAR_SYSTEM = generate_number_system("transfer_far_v2", difficulty=2)@kbench.task(name="learning_transfer")def learning_transfer(llm) -> float:    """    Near vs. Far Transfer Benchmark.    Train on one rule system, then test transfer to:    1. Identical: same system, new test items (baseline)    2. Near: same domain, different surface features    3. Far: different domain, structurally similar rules    Score = 0.30 * identical_acc + 0.35 * near_acc + 0.35 * far_acc    Transfer ratio = far_acc / identical_acc measures genuine generalization.    """    # Build training prompt (10 examples from training system)    training_examples = TRAIN_SYSTEM.examples[:10]    train_block = f"**Rule System: {TRAIN_SYSTEM.name}**\n\n"    train_block += f"Description: {TRAIN_SYSTEM.description}\n\n"    train_block += "**Rules:**\n"    for r in TRAIN_SYSTEM.rules:        train_block += f"- {r}\n"    train_block += "\n**Examples:**\n"    for ex in training_examples:        train_block += f"  Input: {ex['input']}  →  Output: {ex['output']}\n"    results = {}    # ── Condition 1: Identical (same system, held-out test items) ──    condition_results = []    for ti, test_item in enumerate(TRAIN_SYSTEM.test_items):        with kbench.chats.new(f"identical_{ti}"):            prompt = (                train_block +                f"\nApply the rules to this new input:\n"                f"Input: {test_item['input']}\n\n"                f"Respond with ONLY: {{\"answer\": \"<output>\", \"reasoning\": \"<steps>\"}}"            )            try:                result = llm.prompt(prompt, schema=TransferAnswer)                answer = result.answer            except Exception:                raw = llm.prompt(prompt)                try:                    parsed = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())                    answer = str(parsed.get("answer", raw))                except Exception:                    answer = raw            condition_results.append(check_output(answer, test_item["output"]))    results["identical"] = sum(condition_results) / len(condition_results)    # ── Condition 2: Near Transfer (same domain, different specifics) ──    # Show training system, then test on NEAR system items with NEAR rules revealed    near_block = f"\n\n**New Rule System: {NEAR_SYSTEM.name}**\n"    near_block += f"Description: {NEAR_SYSTEM.description}\n\n"    near_block += "**Rules:**\n"    for r in NEAR_SYSTEM.rules:        near_block += f"- {r}\n"    near_block += "\n(No examples provided — use your understanding from the previous system.)\n"    condition_results = []    for ti, test_item in enumerate(NEAR_SYSTEM.test_items):        with kbench.chats.new(f"near_{ti}"):            prompt = (                f"You previously learned a rule system. Now apply a similar but different system.\n\n"                f"**Previous system for reference:**\n{train_block}\n"                f"{near_block}\n"                f"Apply the NEW rules to:\nInput: {test_item['input']}\n\n"                f"Respond with ONLY: {{\"answer\": \"<output>\", \"reasoning\": \"<steps>\"}}"            )            try:                result = llm.prompt(prompt, schema=TransferAnswer)                answer = result.answer            except Exception:                raw = llm.prompt(prompt)                try:                    parsed = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())                    answer = str(parsed.get("answer", raw))                except Exception:                    answer = raw            condition_results.append(check_output(answer, test_item["output"]))    results["near"] = sum(condition_results) / len(condition_results)    # ── Condition 3: Far Transfer (different domain) ──    far_block = f"\n\n**New Rule System: {FAR_SYSTEM.name}**\n"    far_block += f"Description: {FAR_SYSTEM.description}\n\n"    far_block += "**Rules:**\n"    for r in FAR_SYSTEM.rules:        far_block += f"- {r}\n"    far_block += "\n(No examples provided — use your general learning ability.)\n"    condition_results = []    for ti, test_item in enumerate(FAR_SYSTEM.test_items):        with kbench.chats.new(f"far_{ti}"):            prompt = (                f"You previously learned a symbol transformation system. "                f"Now apply a completely different kind of rule system.\n\n"                f"{far_block}\n"                f"Apply the rules to:\nInput: {test_item['input']}\n\n"                f"Respond with ONLY: {{\"answer\": \"<output>\", \"reasoning\": \"<steps>\"}}"            )            try:                result = llm.prompt(prompt, schema=TransferAnswer)                answer = result.answer            except Exception:                raw = llm.prompt(prompt)                try:                    parsed = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())                    answer = str(parsed.get("answer", raw))                except Exception:                    answer = raw            condition_results.append(check_output(answer, test_item["output"]))    results["far"] = sum(condition_results) / len(condition_results)    # ── Compute Metrics ──    identical = results["identical"]    near = results["near"]    far = results["far"]    # Transfer ratios    near_ratio = near / identical if identical > 0 else 0    far_ratio = far / identical if identical > 0 else 0    score = round(0.30 * identical + 0.35 * near + 0.35 * far, 4)    # ── Logging ──    print(f"\n{'='*60}")    print(f"NEAR VS. FAR TRANSFER BENCHMARK RESULTS")    print(f"{'='*60}")    print(f"\n--- Transfer Performance ---")    print(f"Identical (baseline): {identical:.2%}")    print(f"Near transfer:        {near:.2%}  (ratio: {near_ratio:.2f})")    print(f"Far transfer:         {far:.2%}  (ratio: {far_ratio:.2f})")    print(f"\n--- Transfer Gradient ---")    gradient = [identical, near, far]    for i, (label, val) in enumerate(zip(["Identical", "Near", "Far"], gradient)):        bar = "█" * int(val * 30)        print(f"  {label:10s}: {val:.2%} {bar}")    print(f"\nComposite score: {score:.4f}")    return score# ─── Run ────────────────────────────────────────────────────────────learning_transfer.run(llm=kbench.llm)